# Weird AI Attention Mechanism Exploration

In this notebook, you will manually compute attention scores, attention weights, and context vectors before implementing the same ideas in `attention.py`.

The goal is to understand the math behind attention before hiding it inside reusable PyTorch classes.

## Section 1: Create Toy Embeddings

Use a small manually created tensor.
 - Each row represents one token embedding. Each column represents one feature of that token.
 - For this notebook, these are hard-coded embeddings. Later, Weird AI will learn embeddings from lyric tokens.

Example:
```python
import torch

torch.manual_seed(123)

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
    ]
)

print(inputs)
print(inputs.shape)
```

In [17]:

import torch

torch.manual_seed(123)

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
    ]
)

print(inputs)
print(inputs.shape)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300]])
torch.Size([4, 3])


## Section 2: Choose One Query Token
Start by calculating attention for one token.

The selected token acts as the query.  We are asking: "How much should this token pay attention to each token in the sequence?"

In [28]:
query_index = 1
query = inputs[query_index]

print(query)
print(query.shape)

tensor([0.5500, 0.8700, 0.6600])
torch.Size([3])


## Section 3: Compute Attention Scores

Compute dot products between one query token and all input tokens.
 - Conceptual question: Why does a dot product help estimate similarity between two token vectors?

 Note: The attention scores are raw similarity values. Larger dot products suggest stronger similarity between the query token and another token.

In [29]:
attention_scores = torch.empty(inputs.shape[0])

for index, token_embedding in enumerate(inputs):
    attention_scores[index] = torch.dot(query, token_embedding)

print(attention_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434])


In [30]:
# Here is a vectorized version
attention_scores_vectorized = inputs @ query

print(attention_scores_vectorized)

tensor([0.9544, 1.4950, 1.4754, 0.8434])


## Section 4: Normalize with Softmax

Convert raw attention scores into attention weights.
 - Be sure to verify that the sum of the weights is approximately 1.0

In [31]:
attention_weights = torch.softmax(attention_scores, dim=0)

print(attention_weights)
print(attention_weights.sum())

tensor([0.1888, 0.3242, 0.3179, 0.1690])
tensor(1.0000)


## Section 5: Compute a Context Vector

The context vector is a weighted mixture of the input token embeddings.

Tokens with larger attention weights contribute more to the final context vector.

In [32]:
context_vector = torch.zeros(query.shape)

for index, token_embedding in enumerate(inputs):
    context_vector += attention_weights[index] * token_embedding

print(context_vector)

tensor([0.4779, 0.6787, 0.6413])


In [33]:
# Vectorize the context_vector
context_vector_vectorized = attention_weights @ inputs

print(context_vector_vectorized)

tensor([0.4779, 0.6787, 0.6413])


## Section 6: Compute Self-Attention for All Tokens

Instead of calculating attention for only one token, we can calculate attention for every token at once.

This is the basic idea behind self-attention.

In [34]:
all_attention_scores = inputs @ inputs.T

print(all_attention_scores)
print(all_attention_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753],
        [0.9544, 1.4950, 1.4754, 0.8434],
        [0.9422, 1.4754, 1.4570, 0.8296],
        [0.4753, 0.8434, 0.8296, 0.4937]])
torch.Size([4, 4])


In [35]:
all_attention_weights = torch.softmax(all_attention_scores, dim=-1)

print(all_attention_weights)
print(all_attention_weights.sum(dim=-1))

tensor([[0.2863, 0.2737, 0.2704, 0.1695],
        [0.1888, 0.3242, 0.3179, 0.1690],
        [0.1897, 0.3233, 0.3174, 0.1695],
        [0.2046, 0.2956, 0.2915, 0.2084]])
tensor([1.0000, 1.0000, 1.0000, 1.0000])


In [43]:


all_context_vectors = all_attention_weights @ inputs

print(all_context_vectors)
print(all_context_vectors.shape)

tensor([[0.4651, 0.6093, 0.6645],
        [0.4779, 0.6787, 0.6413],
        [0.4776, 0.6779, 0.6413],
        [0.4625, 0.6565, 0.6325]])
torch.Size([4, 3])


## Connecting This Notebook to attention.py

The notebook manually calculated attention step by step.

The `SimpleSelfAttention` class in `attention.py` should perform the same operation in reusable form.

In [45]:
from weird_ai.attention import SimpleSelfAttention

attention = SimpleSelfAttention()

context_vectors, weights = attention(inputs)

print(context_vectors)
print(weights)
print(context_vectors.shape)
print(weights.shape)

tensor([[0.4651, 0.6093, 0.6645],
        [0.4779, 0.6787, 0.6413],
        [0.4776, 0.6779, 0.6413],
        [0.4625, 0.6565, 0.6325]])
tensor([[0.2863, 0.2737, 0.2704, 0.1695],
        [0.1888, 0.3242, 0.3179, 0.1690],
        [0.1897, 0.3233, 0.3174, 0.1695],
        [0.2046, 0.2956, 0.2915, 0.2084]])
torch.Size([4, 3])
torch.Size([4, 4])


## Reflection Questions

1. What does a larger attention score mean?
2. Why do attention weights need to sum to 1?
3. Why is the context vector a weighted mixture of token embeddings?
4. How does self-attention differ from calculating attention for only one query token?
5. How does this connect to lyric generation in Weird AI?